# AndinaLog 03B · Notebook 1 · Diagnóstico IoT didáctico

**Objetivo:** conservar las diez columnas Bronze y añadir, junto a cada una, `*_estado` y `*_motivo`. No requiere catálogo JSON ni motor externo. Exporta solamente `diagnosticado` y `cuarentena`.

`OK` significa que no se detectó un problema en esa columna; `REVISAR` indica una advertencia que no bloquea por sí sola; `CRITICO` bloquea el uso de la fila bajo esta política; `NO_EVALUABLE` indica que falta contexto para una comprobación. `en_cuarentena` corresponde a la **fila completa**, no a una sola columna.

El docente confirmó que las fechas sin zona del caso representan hora de Bolivia (`America/La_Paz`). Este diagnóstico conserva el texto Bronze; la conversión a UTC corresponde al tratamiento.


In [1]:
from pathlib import Path
import hashlib
import sys
import pandas as pd

ENTORNO = "auto"  # auto, local, drive
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-IOT-diagnostico-didactico-v1"
ZONA_HORARIA_ORIGEN = "America/La_Paz"
COLUMNAS_BRONZE = [
    "timestamp", "viaje_id", "order_id", "camion_id", "producto_id",
    "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct",
    "desviacion_termica_flag", "desviacion_proximos_60min_flag",
]

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
        if not (raiz / "datasets/AndinaLog_03B_Bronce/andinalog_iot_telemetry.csv").is_file():
            raise FileNotFoundError(raiz)
        return raiz
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets/AndinaLog_03B_Bronce/andinalog_iot_telemetry.csv").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

RAIZ = encontrar_raiz()
RUTA_BRONZE = RAIZ / "datasets/AndinaLog_03B_Bronce/andinalog_iot_telemetry.csv"
RUTA_PRODUCTOS = RAIZ / "proyecto-integrador/andinalog_productos/notebook2/salidas/andinalog_productos_silver.csv"
RUTA_FLOTA = RAIZ / "proyecto-integrador/andinalog_flota/notebook2/salidas/andinalog_flota_silver.csv"
SALIDAS = RAIZ / "proyecto-integrador/01_diagnostico/andinalog_iot_telemetry/salidas"
HASH_BRONZE = hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()

bronze = pd.read_csv(RUTA_BRONZE, dtype="string", encoding="utf-8-sig", keep_default_na=False)
if list(bronze.columns) != COLUMNAS_BRONZE:
    raise ValueError(f"Esquema Bronze inesperado: {list(bronze.columns)}")
df = bronze.copy(deep=True)
df.insert(0, "fila_bronze", range(1, len(df)+1))
print("Lecturas Bronze:", len(df))


Lecturas Bronze: 28920


## Reglas visibles de diagnóstico

Cada máscara añade un estado y un motivo. Una misma columna puede acumular varios motivos.


In [2]:
# Las reglas están visibles aquí, sin depender de un catálogo externo.
PRIORIDAD = {"OK":0, "NO_EVALUABLE":1, "REVISAR":2, "CRITICO":3}
for columna in COLUMNAS_BRONZE:
    df[f"{columna}_estado"] = "OK"
    df[f"{columna}_motivo"] = ""

def marcar(columna, mascara, estado, motivo):
    mascara = pd.Series(mascara, index=df.index).fillna(False).astype(bool)
    e, m = f"{columna}_estado", f"{columna}_motivo"
    subir = mascara & df[e].map(PRIORIDAD).lt(PRIORIDAD[estado])
    df.loc[subir, e] = estado
    previo = df.loc[mascara, m]
    df.loc[mascara, m] = previo.where(previo.eq(""), previo + "; ") + motivo

def no_vacio(columna):
    return df[columna].str.strip().ne("")

# 1. Completitud y formato de fecha e identificadores.
for columna in COLUMNAS_BRONZE:
    marcar(columna, ~no_vacio(columna), "CRITICO", "Valor faltante")

fecha = pd.to_datetime(df["timestamp"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
forma_fecha = df["timestamp"].str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}").fillna(False)
marcar("timestamp", no_vacio("timestamp") & (~forma_fecha | fecha.isna()), "CRITICO", "Formato o fecha de calendario inválidos")

for columna, patron in {
    "viaje_id":r"VIA-\d{5}", "order_id":r"ORD-\d{4}-\d{5}",
    "camion_id":r"CAM-\d{2}", "producto_id":r"PROD-\d{3}",
}.items():
    coincide = df[columna].str.fullmatch(patron).fillna(False)
    marcar(columna, no_vacio(columna) & ~coincide, "CRITICO", f"Formato esperado: {patron}")

# 2. Magnitudes y unidades según el dominio de cadena de frío.
temperatura = pd.to_numeric(df["temperatura_cabina_c"], errors="coerce")
humedad = pd.to_numeric(df["humedad_cabina_pct"], errors="coerce")
marcar("temperatura_cabina_c", no_vacio("temperatura_cabina_c") & temperatura.isna(), "CRITICO", "Temperatura no numérica")
marcar("temperatura_cabina_c", temperatura.eq(-999), "CRITICO", "-999 es centinela: no es una temperatura")
marcar("humedad_cabina_pct", no_vacio("humedad_cabina_pct") & humedad.isna(), "CRITICO", "Humedad no numérica")
marcar("humedad_cabina_pct", humedad.notna() & ~humedad.between(0,100), "CRITICO", "Humedad fuera de 0 a 100 %")

unidad = df["temp_unit"].str.strip().str.upper()
marcar("temp_unit", unidad.eq("F"), "REVISAR", "Fahrenheit reconocido; convertir a Celsius en tratamiento")
marcar("temp_unit", unidad.eq("K"), "CRITICO", "Kelvin no es unidad operacional esperada en AndinaLog")
marcar("temp_unit", no_vacio("temp_unit") & ~unidad.isin(["C","F","K"]), "CRITICO", "Unidad de temperatura desconocida")
marcar("temperatura_cabina_c", unidad.eq("K") & temperatura.notna() & temperatura.ne(-999), "NO_EVALUABLE", "Rango térmico no evaluado con unidad K")

for columna in ("desviacion_termica_flag", "desviacion_proximos_60min_flag"):
    marcar(columna, no_vacio(columna) & ~df[columna].isin(["0","1"]), "CRITICO", "Bandera distinta de 0 o 1")

# 3. Clave candidata: una lectura por viaje y timestamp.
firma = pd.util.hash_pandas_object(df[COLUMNAS_BRONZE], index=False)
variantes = firma.groupby([df["viaje_id"], df["timestamp"]], dropna=False).transform("nunique")
clave_repetida = df.duplicated(["viaje_id","timestamp"], keep=False)
copia = clave_repetida & variantes.eq(1) & df.duplicated(COLUMNAS_BRONZE, keep="first")
conflicto = clave_repetida & variantes.gt(1)
for columna in ("viaje_id", "timestamp"):
    marcar(columna, copia, "CRITICO", "Copia idéntica de la lectura; no contar dos veces")
    marcar(columna, conflicto, "CRITICO", "Misma clave con datos diferentes; revisar ambas lecturas")

# 4. Cobertura parcial de dimensiones Silver; ausencia es advertencia, no prueba de ID inválido.
productos = None
if RUTA_PRODUCTOS.is_file():
    productos = pd.read_csv(RUTA_PRODUCTOS, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    if productos["producto_id"].duplicated().any():
        raise ValueError("Productos Silver tiene producto_id duplicado")
    marcar("producto_id", no_vacio("producto_id") & ~df["producto_id"].isin(productos["producto_id"]),
           "REVISAR", "Sin correspondencia en Productos Silver de cobertura parcial")

if RUTA_FLOTA.is_file():
    flota = pd.read_csv(RUTA_FLOTA, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    if flota["camion_id"].duplicated().any():
        raise ValueError("Flota Silver tiene camion_id duplicado")
    camion_formato_valido = df["camion_id"].str.fullmatch(r"CAM-\d{2}").fillna(False)
    marcar("camion_id", camion_formato_valido & ~df["camion_id"].isin(flota["camion_id"]),
           "REVISAR", "Sin correspondencia en Flota Silver de cobertura parcial")

# 5. Coherencia entre bandera térmica y rango del producto. Una excursión correcta no es error.
if productos is not None:
    p = productos.set_index("producto_id")
    objetivo = pd.to_numeric(df["producto_id"].map(p["temperatura_conservacion_requerida_c"]), errors="coerce")
    tolerancia = pd.to_numeric(df["producto_id"].map(p["tolerancia_temperatura_c"]), errors="coerce")
    temp_c = temperatura.where(unidad.eq("C"), (temperatura-32)*5/9)
    evaluable = (unidad.isin(["C","F"]) & temperatura.notna() & temperatura.ne(-999)
                 & objetivo.notna() & tolerancia.notna() & tolerancia.ge(0)
                 & df["desviacion_termica_flag"].isin(["0","1"]))
    fuera = (temp_c-objetivo).abs() > tolerancia
    incoherente = evaluable & fuera.ne(df["desviacion_termica_flag"].eq("1"))
    marcar("desviacion_termica_flag", incoherente, "REVISAR", "Flag no coincide con objetivo y tolerancia del producto")

print("Reglas aplicadas; no se modificaron columnas Bronze")


Reglas aplicadas; no se modificaron columnas Bronze


## Resumen y comprobaciones

`cantidad_problemas` cuenta columnas con estado distinto de `OK`; no es el número de reglas activadas.


In [3]:
# Estado por fila: un CRITICO envía la fila a cuarentena.
estados = [f"{c}_estado" for c in COLUMNAS_BRONZE]
motivos = [f"{c}_motivo" for c in COLUMNAS_BRONZE]
df["cantidad_problemas"] = df[estados].ne("OK").sum(axis=1)
df["en_cuarentena"] = df[estados].eq("CRITICO").any(axis=1)
df["severidad_maxima"] = "OK"
df.loc[df[estados].eq("NO_EVALUABLE").any(axis=1), "severidad_maxima"] = "NO_EVALUABLE"
df.loc[df[estados].eq("REVISAR").any(axis=1), "severidad_maxima"] = "REVISAR"
df.loc[df[estados].eq("CRITICO").any(axis=1), "severidad_maxima"] = "CRITICO"

def resumen_fila(fila):
    columnas = [c for c in COLUMNAS_BRONZE if fila[f"{c}_estado"] != "OK"]
    textos = []
    for c in columnas:
        for motivo in fila[f"{c}_motivo"].split("; "):
            texto = f"{c}: {motivo}"
            if motivo and texto not in textos:
                textos.append(texto)
    return pd.Series({"columnas_con_problemas":"|".join(columnas),"motivos_fila":" | ".join(textos)})

df[["columnas_con_problemas","motivos_fila"]] = df.apply(resumen_fila, axis=1)
df["zona_horaria_origen"] = ZONA_HORARIA_ORIGEN
df["version_diagnostico"] = VERSION_DIAGNOSTICO
df["sha256_bronze"] = HASH_BRONZE
cuarentena = df.loc[df["en_cuarentena"]].copy()

pd.testing.assert_frame_equal(df[COLUMNAS_BRONZE], bronze)
assert len(df) == len(bronze)
assert cuarentena["motivos_fila"].ne("").all()
assert len(cuarentena) == int(df["en_cuarentena"].sum())
assert hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest() == HASH_BRONZE
print("Diagnosticado:", len(df), "| Con problemas:", int(df["cantidad_problemas"].gt(0).sum()),
      "| Cuarentena:", len(cuarentena))
display(df[["fila_bronze","temperatura_cabina_c","temperatura_cabina_c_estado",
            "temperatura_cabina_c_motivo","temp_unit_estado","temp_unit_motivo",
            "en_cuarentena","motivos_fila"]].head(10))


Diagnosticado: 28920 | Con problemas: 7539 | Cuarentena: 505


,fila_bronze,temperatura_cabina_c,temperatura_cabina_c_estado,temperatura_cabina_c_motivo,temp_unit_estado,temp_unit_motivo,en_cuarentena,motivos_fila
0,1,2.19,OK,,OK,,False,camion_id: Sin correspondencia en Flota Silver...
1,2,4.91,OK,,OK,,False,camion_id: Sin correspondencia en Flota Silver...
2,3,2.76,OK,,OK,,False,camion_id: Sin correspondencia en Flota Silver...
3,4,3.96,OK,,OK,,False,camion_id: Sin correspondencia en Flota Silver...
4,5,4.49,OK,,OK,,False,camion_id: Sin correspondencia en Flota Silver...
5,6,2.29,OK,,OK,,False,camion_id: Sin correspondencia en Flota Silver...
6,7,3.76,OK,,OK,,False,camion_id: Sin correspondencia en Flota Silver...
7,8,3.48,OK,,OK,,True,camion_id: Sin correspondencia en Flota Silver...
8,9,2.85,OK,,OK,,False,camion_id: Sin correspondencia en Flota Silver...
9,10,3.57,OK,,OK,,False,camion_id: Sin correspondencia en Flota Silver...


## Exportación

Solo se generan dos CSV para esta versión didáctica.


In [4]:
# Dos archivos de salida; el Bronze permanece intacto.
SALIDAS.mkdir(parents=True, exist_ok=True)
ruta_diagnosticado = SALIDAS / "andinalog_iot_telemetry_didactico_v1_diagnosticado.csv"
ruta_cuarentena = SALIDAS / "andinalog_iot_telemetry_didactico_v1_cuarentena.csv"
df.to_csv(ruta_diagnosticado, index=False, encoding="utf-8-sig")
cuarentena.to_csv(ruta_cuarentena, index=False, encoding="utf-8-sig")
assert hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest() == HASH_BRONZE
print(ruta_diagnosticado)
print(ruta_cuarentena)


c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\01_diagnostico\andinalog_iot_telemetry\salidas\andinalog_iot_telemetry_didactico_v1_diagnosticado.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\01_diagnostico\andinalog_iot_telemetry\salidas\andinalog_iot_telemetry_didactico_v1_cuarentena.csv


## Interpretación y siguiente etapa

El diagnóstico es una **tabla explicada**: cada variable tiene estado y motivo, mientras `en_cuarentena` decide el destino de la fila completa. Las advertencias de cobertura parcial no equivalen a datos inválidos. Fahrenheit queda para conversión en el Notebook 2; Kelvin y `-999` no se normalizan silenciosamente. El tratamiento debe leer esta nueva estructura directamente; la versión técnica anterior del tratamiento, que lee `incidencias_json`, **no es compatible sin adaptación**.

La zona horaria de origen fue confirmada como hora de Bolivia para fechas sin sufijo. Este notebook no transforma el timestamp Bronze; el tratamiento puede derivar UTC y conservar el original.
